# Full PCP proposal — third A100 worker

Run this alongside workers 0 and 1 in a **third separate A100 runtime**. Add the same Supabase Secrets and Run all. Code and frozen checkpoints download automatically and are SHA256-verified. This has its own experiment, `smolvla-pcp-full-proposal-jeff-matched-v4`, and does not alter running workers.

**Algorithm at each action chunk:** predict a clean chunk at early/middle/late sampler indices 2/5/8 (80%/50%/20% noise remaining); use three independently re-noised one-step predictions to measure action disagreement; optionally gate on standardized RMS disagreement >= .06; apply one normalized Q-gradient correction inside radius .06 to the first 10 actions; re-noise into three independent branches; complete the remaining flow integration for each branch; rank their fully denoised chunks by Q; execute the best. The exact uncorrected continuation is included as candidate zero and is used when the gate is closed. No backpropagation through the flow and no model fine-tuning.

**18 fixed configurations:** frozen MC/continued TD x early/middle/late correction x always-on/disagreement-gated (12); reranking without correction for each critic at each timing (6). Radius is fixed at .06 to focus replication on timing. The exact parent P&P schedule and random draws are retained in the baseline continuation at every timing. CNN is excluded because prior directional evidence was weaker. All arms receive equal exposure: up to 2,000 starting states x three seeds = 6,000 full episodes each, plus the **stock SmolVLA baseline (no P&P or Q correction)**. The treatments and their candidate-zero fallback retain P&P; the fallback is distinct from the stock evaluation control. No time cutoff or adaptive promotion. This can exceed 12 hours; runtime estimates come from measured throughput.

The .06 uncertainty threshold is a fixed heuristic. Probe disagreement measures local action instability, not calibrated critic uncertainty or failure probability. Regeneration does not guarantee actions remain in distribution; reranking does not guarantee real success. The baseline fallback only enforces nondecreasing **predicted Q** when scores are finite. The correction radius bounds the injected clean-action displacement, not the final regenerated chunk.

Each branch uses the full episode batch on GPU and shares the encoded visual/language prefix; branches are evaluated sequentially to control memory. Parallel simulator processes retain only one environment per active lane. Batch 64 targets the high-RAM A100 runtime; CUDA OOM halves it automatically.

Supabase stores every completed/error episode, executed actions, checkpoint/configuration identity, uncertainty, gate decisions, all terminal candidate Q scores, selected branch, correction magnitude and intervention/search counts. Progress uses `artifacts/smolvla_pcp_sweep/smolvla-pcp-full-proposal-jeff-matched-v4/worker_0_latest.json`. Success changes, rescues/spoils and standard errors group repeated seeds by starting state. Benchmark-state evaluation remains exploratory.


In [ ]:
WORKER_INDEX = 0  # this standalone experiment has one worker (your third Colab)
WORKER_COUNT = 1
BATCH_SIZE = 8
SIMULATOR_PROCESSES = min(8, max(1, (__import__('os').cpu_count() or 2) - 2))
BUNDLE_PATH = ''
BUNDLE_BUCKET = 'artifacts'
BUNDLE_SHA256 = '05af76e02d83398a3b6969c9efacf7032ab45f8452d110cb6f57eede6e5ccf9a'
BUNDLE_STORAGE_KEY = 'smolvla_pcp_sweep/bundles/05af76e02d83398a3b6969c9efacf7032ab45f8452d110cb6f57eede6e5ccf9a/trial_bundle.zip'
CHECKPOINT_SHA256 = {'frozen_mc': '9b8800bc9f27116afcb1812f93f8a9f6767ac012c7cdac790502032fbb7d0c8e', 'cnn_mc': 'afcf832d43ffca44af843e4e5301fa0a84b1b2e485703e2a1dc13aa02b260151', 'td': '1e3fa3f093a711d0651b69e7f152a5b56a7e7a85ec754c45b0afe5834d9f324e'}
OUTPUT_DIR = '/content/pcp-full-proposal'


In [ ]:
import os, sys, subprocess, zipfile
from pathlib import Path
os.environ.update(MUJOCO_GL='egl', NVIDIA_DRIVER_CAPABILITIES='compute,utility,graphics',
                  TOKENIZERS_PARALLELISM='false', OMP_NUM_THREADS='1', OPENBLAS_NUM_THREADS='1')
from google.colab import userdata
for key in ('SUPABASE_URL','SUPABASE_SERVICE_KEY','HF_TOKEN'):
    try:
        value=userdata.get(key)
        if value: os.environ[key]=value
    except Exception: pass
for key in ('SUPABASE_URL','SUPABASE_SERVICE_KEY'):
    if not os.environ.get(key): raise RuntimeError(f'Add {key} in Colab Secrets and allow notebook access.')
if not BUNDLE_PATH:
    subprocess.run([sys.executable,'-m','pip','install','-q','supabase'],check=True)
    from supabase import create_client
    import hashlib
    client=create_client(os.environ['SUPABASE_URL'],os.environ['SUPABASE_SERVICE_KEY'])
    cached=Path('/content')/f'pcp_bundle_{BUNDLE_SHA256}.zip'
    if not cached.exists() or hashlib.sha256(cached.read_bytes()).hexdigest()!=BUNDLE_SHA256:
        data=client.storage.from_(BUNDLE_BUCKET).download(BUNDLE_STORAGE_KEY)
        if hashlib.sha256(data).hexdigest()!=BUNDLE_SHA256:
            raise RuntimeError('Downloaded bundle SHA256 mismatch')
        cached.write_bytes(data)
    BUNDLE_PATH=str(cached)
if __import__('hashlib').sha256(Path(BUNDLE_PATH).read_bytes()).hexdigest()!=BUNDLE_SHA256:
    raise RuntimeError('Bundle differs from the exact version pinned in this notebook')
PACKAGE=Path('/content/smolvla-pcp-trial')
PACKAGE.mkdir(exist_ok=True)
with zipfile.ZipFile(BUNDLE_PATH) as archive:
    if any(Path(name).is_absolute() or '..' in Path(name).parts for name in archive.namelist()):
        raise ValueError('Unsafe bundle path')
    archive.extractall(PACKAGE)
print({'python':sys.version,'executable':sys.executable},flush=True)
subprocess.run([sys.executable,'-m','pip','--version'],check=True)
install=subprocess.run([sys.executable,'-m','pip','install','-e',str(PACKAGE)+'[sim]'],
                       stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
install_log=Path('/content/pcp_dependency_install.log')
install_log.write_text(install.stdout)
if install.returncode:
    print('\n'.join(install.stdout.splitlines()[-120:]),flush=True)
    raise RuntimeError(f'Dependency installation failed (exit {install.returncode}). '
                       f'Full log: {install_log}. Paste the ERROR lines above; no experiment has started.')
print('Simulation/model dependencies installed.',flush=True)
subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=False,stdout=subprocess.DEVNULL)
if str(PACKAGE) not in sys.path: sys.path.insert(0,str(PACKAGE))
from pnp import env_setup
if env_setup._find_nvidia_egl_library() is None:
    import re
    driver=subprocess.check_output(['nvidia-smi','--query-gpu=driver_version','--format=csv,noheader'],text=True).splitlines()[0]
    major=re.match(r'\d+',driver.strip()).group()
    subprocess.run(['apt-get','update','-qq'],check=True)
    subprocess.run(['apt-get','install','-y','--no-install-recommends',f'libnvidia-gl-{major}'],check=True)
env_setup._ensure_nvidia_egl_vendor()
if env_setup._find_nvidia_egl_library() is None: raise RuntimeError('NVIDIA EGL missing; hardware rendering is required.')
env_setup._ensure_libero_config()
env_setup._fix_torch_quant_compat()
env_setup._remove_broken_optional_torchaudio()
env_setup._ensure_libero_assets(token=os.getenv('HF_TOKEN'))
import torch, hashlib, json
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime.')
metadata=json.loads((PACKAGE/'bundle_manifest.json').read_text())
if metadata['checkpoints']!=CHECKPOINT_SHA256: raise RuntimeError('Unexpected checkpoint manifest')
CHECKPOINT_PATHS={name:str(PACKAGE/'checkpoints'/f'{name}.pt') for name in metadata['checkpoints']}
for name,path in CHECKPOINT_PATHS.items():
    if hashlib.sha256(Path(path).read_bytes()).hexdigest()!=metadata['checkpoints'][name]:
        raise RuntimeError(f'Checkpoint mismatch: {name}')
print({'gpu':torch.cuda.get_device_name(0),'batch_size':BATCH_SIZE,'worker':WORKER_INDEX,'simulator_processes':SIMULATOR_PROCESSES})


In [ ]:
from pnp.smolvla_jeff_replication import require_replication
print(require_replication())
from pnp.smolvla_pcp_full_proposal import run_full_proposal
report=run_full_proposal(checkpoint_paths=CHECKPOINT_PATHS, worker_index=WORKER_INDEX,
                        worker_count=WORKER_COUNT, batch_size=BATCH_SIZE,
                        simulator_processes=SIMULATOR_PROCESSES, output_dir=OUTPUT_DIR)
